In [2]:
!pip install pymysql


  Obtaining dependency information for pymysql from https://files.pythonhosted.org/packages/e5/30/20467e39523d0cfc2b6227902d3687a16364307260c75e6a1cb4422b0c62/PyMySQL-1.1.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.4 MB/s eta 0:00:00


In [4]:
!pip install boto3

  Obtaining dependency information for boto3 from https://files.pythonhosted.org/packages/3f/57/863c7add6a601d22f9bf013e8e57a48d637178d05d4f0f213fe6572392ca/boto3-1.34.98-py3-none-any.whl.metadata
  Obtaining dependency information for botocore<1.35.0,>=1.34.98 from https://files.pythonhosted.org/packages/68/8a/3556b87d76671c203b49584ee10d1609ec597f0533151bc668041b7ca69b/botocore-1.34.98-py3-none-any.whl.metadata
  Obtaining dependency information for s3transfer<0.11.0,>=0.10.0 from https://files.pythonhosted.org/packages/83/37/395cdb6ee92925fa211e55d8f07b9f93cf93f60d7d4ce5e66fd73f1ea986/s3transfer-0.10.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 1.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 4.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: botocore
    Found existing installation: botocore 1.29.76
    Uninstalling botocor

In [13]:
import os
import pathlib
import datatier
import awsutil
import uuid
import sys
from configparser import ConfigParser
import pandas as pd
import math 


# eliminate traceback so we just get error message:
sys.tracebacklimit = 0

#
# what config file should we use for this session?
#
config_file = 'config.ini'

s3_profile = 's3readwrite'

os.environ['AWS_SHARED_CREDENTIALS_FILE'] = config_file

# boto3.setup_default_session(profile_name=s3_profile)

configur = ConfigParser()
configur.read(config_file)
# bucketname = configur.get('s3', 'bucket_name')

# s3 = boto3.resource('s3')
# bucket = s3.Bucket(bucketname)

#
# now let's connect to our RDS MySQL server:
#
endpoint = configur.get('rds', 'endpoint')
portnum = int(configur.get('rds', 'port_number'))
username = configur.get('rds', 'user_name')
pwd = configur.get('rds', 'user_pwd')
dbname = configur.get('rds', 'db_name')

dbConn = datatier.get_dbConn(endpoint, portnum, username, pwd, dbname)

if dbConn is None:
  print('**ERROR: unable to connect to database, exiting')
  sys.exit(0)



In [14]:
import pandas as pd

def read_csv_file(file_path):
    df = pd.read_csv(file_path)

    for index, row in df.iterrows():
        # Check for NaN values and replace them with None
        Artist = row['Artist']
        if pd.isna(Artist): 
            Artist = None
        decade = row['Decade']
        if pd.isna(decade): 
            decade = None
        genre = row['Genre']
        if pd.isna(genre): 
            genre = None
        lyrix = row['Lyrix']
        if pd.isna(lyrix): 
            lyrix = None
        song_name = row['Song name']
        if pd.isna(song_name): 
            song_name = None
        year = row['Year']
        if pd.isna(year): 
            year = None
        potpourri = row['Songs Potpourri']
        if pd.isna(potpourri): 
            potpourri = None
        Link = row['Youtube Link']
        if pd.isna(Link): 
            Link = None

        # Print the data to be inserted
        print([Artist, decade, genre, lyrix, song_name, year, potpourri, Link])

        # Insert data into SQL table
        sql = "INSERT INTO songs (artist, decade, genre, lyrix, song_name, song_year, songs_potpourri, youtube_link) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)"
        datatier.perform_action(dbConn, sql, [Artist, decade, genre, lyrix, song_name, year, potpourri, Link])

# Example usage:
read_csv_file("songs_clean.csv")


['Bob Dylan (as sung by Patti Smith at 1996 Nobel Prize ceremony)', '1960s', 'Folk/Protest', "Oh, where have you been, my blue-eyed son? Oh, where have you been, my darling young one? I've stumbled on the side of twelve misty mountains. I've walked and I've crawled on six crooked highways. I've stepped in the middle of seven sad forests. I've been out in front of a dozen dead oceans. I've been ten thousand miles in the mouth of a graveyard.", "A Hard Rain's Gonna Fall", '1963', None, 'https://youtu.be/941PHEJHCwU?t=8']
['Aliotta Haynes & Jeremiah (Skip Haynes)', '1970s', 'Folk/Protest', "It starts up north from Hollywood, water on the driving side, Concrete mountains rearing up, throwing shadows just about five, Sometimes you can smell the green if your mind is feeling fine, There ain't no finer place to be, than running LSD, there's no deeper peace of mind, or place you see.", 'Lake Shore Drive', '1971', None, 'https://www.youtube.com/watch?v=0saZiLV7-7E#t=1m19s']
['Dolly Parton Also 

['Sonny Bono / Sung by Sonny and Cher Played in Groundhog Day', '1960s', 'Pop', "I got flowers in the spring I got you to wear my ring And when I'm sad, you're a clown And if I get scared, you're always around Don't let them say your hair's too long 'Cause I don't care, with you I can't go wrong Then put your little hand in mine There ain't no hill or mountain we can't climb", 'I Got You Babe', '1965', None, 'https://www.youtube.com/watch?v=BERd61bDY7k#t=0m59s. https://youtu.be/OyBSrBqogPY?t=1']
["Puff Daddy & The Family (with The Notorious B.I.G., Lil' Kim, & The Lox)", '1990s', 'Hip Hop/Rap', "Swimming in women with they own condominiums\nFive plus fives, who drive Millenniums\nIt's all about the Benjamins, what\nI get a fifty pound bag of eu for the mutts\nFive carats on my hands with the cuts\nAnd something European chromed out with the clutch", "It's All About the Benjamins", '1997', None, 'https://youtu.be/0c58ppLPJcQ?t=62']
['Lauryn Hill', '1990s', 'Jazz/Blues/RnB', "The second 

['Hoyt Axton (sung by the Kingston Trio)', '1960s', 'Folk/Protest', 'When I was a little baby, my momma said, "Hey son. Travel where you will and grow to be a man and sing what must be sung, poor boy. Sing what must be sung."...Now that I\'m a grown man, I\'ve travelled here and there. I\'ve learned that a bottle of brandy and a song, the only ones who ever cared, poor boy, the only ones who ever cared.', 'King of the Road', '1962', None, 'https://youtu.be/t_Lo40O_zgA?t=45']
['Stephen Sondheim (as sung by Judy Collins)', '1970s', 'Folk/Protest', "Just when I'd stopped opening doors, Finally knowing the one that I wanted was yours, Making my entrance again with my usual flair, Sure of my lines, No one is there, Don't you love farce? My fault, I fear I thought that you'd want what I want Sorry, my dear!", 'Send In The Clowns', '1973', None, 'https://www.youtube.com/watch?v=8L6KGuTr9TI#t=1m24s']
['Bayside Boys Los Del Rio', '1990s', 'Pop', "Don't you worry about my boyfriend/He's a boy wh

['Mulan (as sung by Donny Osmond)', '1990s', 'Musical/Movie', "I'm never gonna catch my breath. Say goodbye to those who knew me. Boy, was I a fool in school for cutting gym. This guy's got 'em scared to death. Hope he doesn't see right through me. Now I really wish that I knew how to swim", 'I’ll make a man out of you / Mulan`', '1998', None, 'https://www.youtube.com/watch?v=TVcLIfSC4OE#t=1m15s']
['Mulan Stephen Schwartz, Matthew Wilder and David Zippel As Sung by Donny Osmond', '1990s', 'Musical/Movie', "I'm never gonna catch my breath!\nSay goodbye to those who knew me!\nBoy, was I a fool in school for cutting gym?\nThis guy got 'em scared to death!\nHope he doesn't see right through me!\nNow I really wish that I knew how to swim!", 'I’ll make a man out of you / Mulan\n', '1998', None, None]
['Maroon 5', '2000s', 'Pop', "Beauty queen of only eighteen, she had some trouble with herself\nHe was always there to help her, she always belonged to someone else\nI drove for miles and miles,

['Robbie Robertson/The Band', '1960s', 'Rock', 'I pulled into Nazareth, was feelin\' about half past dead.\nI just need some place where I can lay my head.\n"Hey, mister, can you tell me where a man might find a bed?"\nHe just grinned and shook my hand, "No," was all he said,\nTake a load off Fanny\nTake a load for free\nTake a load off Fanny\nAnd (and) (and) you put the load right on me.', 'The Weight', '1968', None, 'https://youtu.be/FFqb1I-hiHE?t=11']
['Don McLean', '1970s', 'Folk/Protest', "February made me shiver With every paper I'd deliver\nBad news on the doorstep I couldn't take one more step\nI can't remember if I cried When I read about his widowed bride\nSomething touched me deep inside", None, '1971', None, 'https://youtu.be/RciM7P9K3FA?t=28']
['Bruce Springsteen', '1980s', 'Rock', 'In \'65 tension was running high at my high school\nThere was a lot of fights \'tween the black and white\nThere was nothing you could do\nTwo cars at a light on a Saturday night\nIn the back s

...

['Carla Thomas and Otis Redding', '1960s', 'Jazz/Blues/RnB', "I know we can do it Carla\nI'm gonna keep my promises\nI'm gonna hold on that we can do it, baby\nOh, it's not too late\nYou're gonna love me, nobody else\nOh, Otis let's finish what we started\nLove me, talk holding me\nThis is I wanted to give, darling\nThe same for you honey\nAnd call it a new year's resolution", "New Year's Resolution", '1967', None, 'https://youtu.be/WB3i5WO9A6Y?si=TRslKEwCMLWlo0hC&t=166']
['Bing Crosby in Holiday Inn Irving Berlin', 'Vintage', 'Musical/Movie', "One minute to midnight\nOne minute to go\nOne minute to say goodbye\nBefore we say hello Let's start the new year right\nTwelve o'clock tonight\nWhen they dim the light\nLet's begin\nKissing the old year out\nKissing the new year in", "Let's Start The New Year Right", '1942', None, 'https://youtu.be/mFmWurPim-0?si=uSluhiZDICLF3bAk&t=7']
['Jeff Buckley', '1990s', 'Alternative/Indie', "Leave your office, run past your funeral\nLeave your home, car

['Sung by Nat King Cole   eden ahbez', 'Vintage', 'Jazz/Blues/RnB', 'While we spoke of many things\nFools and kings\nThis he said to me:\n"The greatest thing you\'ll ever learn\nIs just to love and be loved in return"', 'Nature Boy', '1948', None, 'https://youtu.be/Iq0XJCJ1Srw?si=kesFtWGSRxWMQF60&t=56']
['Al Green Al Green, Al Jackson Jr & Willie Mitchell', '1970s', 'Jazz/Blues/RnB', "Let me say that since, baby, since we've been together\nOoh, loving you forever\nIs what I need\nLet me, be the one you come running to\nI'll never be untrue", "Let's Stay Together", '1972', None, 'https://youtu.be/uSu6tcbMOu0?si=KIkPVUm8wgFjMg2O&t=46']
['Sung by Nat King Cole. Bert Kaempfert & Milt Gabler', '1960s', 'Jazz/Blues/RnB', "Love is all that I can give to you\nLove is more than just a game for two\nTwo in love can make it, take my heart and please don't break it\nLove was made for me and you", 'L-O-V-E', '1964', None, 'https://youtu.be/JErVP6xLZwg?si=2fc_zYs-xGKDsSXO&t=33']
['Tracy Chapman and 

['Sung by Fred Astaire. George &  Ira Gershwin', 'Vintage', 'Musical/Movie', 'The way you wear your hat\nThe way you sip your tea\nThe memory of all that', "They Can't Take That Away From Me in Shall We Dance", '1937', None, 'https://youtu.be/fuufFgAMkGE?si=dERLOuuoHmncMrHk&t=110']
['Sung here by The McGuire Sisters. Johnny Mercer', 'Vintage', 'Musical/Movie', "When an irrepressible smile such as yours\nWarms an old implacable heart such as mine\nDon't say no because I insist\nSomewhere, somehow\nSomeone's gotta be kissed", "Something's Gotta Give from film Daddy Long Legs", '1955', None, 'https://youtu.be/856Ve-Rxr-c?si=qcAh-TIPaz3YPVcM&t=37']
['Cat Stevens', '1970s', 'Musical/Movie', "Well, I think it's fine\nBuilding jumbo planes\nOr taking a ride on a cosmic train\nSwitch on summer from a slot machine\nYes, get what you want to if you want\n'Cause you can get anything\nI know we've come a long way\nWe're changing day to day\n ", 'Where Do the Children Play? (from Harold and Maude)'

['Cat Stevens', '1970s', 'Folk/Protest', "If I ever lose my eyes\nIf my colors all run dry\nYes, if I ever lose my eyes\nOh if, I won't have to cry no more\nYes, I am bein' followed by a moonshadow\nMoonshadow, moonshadow", 'Moonshadow', '1975', None, 'https://youtu.be/kGNxKnLmOH4?si=1FuT-obCUYUZ-j7H&t=42']
['Sung by Pink Floyd. Roger Waters', '1970s', 'Rock', "Everyone you meet (everyone you meet)\nAnd all that you slight\nAnd everyone you fight\nAnd all that is now\nAnd all that is gone\nAnd all that's to come\nAnd everything under the sun is in tune\nBut the sun is eclipsed by the moon", 'Eclipse', '1973', None, 'https://youtu.be/EeOik4xzp60?si=M48Flj5pNrZL9uW1&t=61']
['Talking Heads. David Byrne, Chris Frantz Jerry Harrison & Tina Weymouth ', '1980s', 'Alternative/Indie', "Hi yo, I got plenty of time\nHi yo, you got light in your eyes\nAnd you're standing here beside me\nI love the passing of time\nNever for money, always for love\nCover us and say goodnight\nSay goodnight", 'This 

['Jack Johnson. Bob Dorough & Jack Johnson', '2000s', 'Folk/Protest', "We've got to learn to\nReduce, reuse, recycle\nWell, if you're going to the market to buy some juice\nYou've got to bring your own bags, and you learn to reduce your waste\nWe gotta learn to reduce", "The 3 R's", '2006', None, 'https://youtu.be/Uu7uujDhcHk?si=J2behoSas6zqrtkM&t=67']
['Talking Heads. David Byrne', '1980s', 'Alternative/Indie', "Once there were parking lots\nNow it's a peaceful oasis\nYou've got it, you've got it\nThis was a Pizza Hut\nNow it's all covered with daisies\nYou got it, you got it...Don't leave me stranded here\nI can't get used to this lifestyle", 'Nothing (but) Flowers', '1988', None, 'https://youtu.be/2twY8YQYDBE?si=8exXcx1p41P01CcM&t=178']
['John Denver', '1990s', 'Folk/Protest', "Celebrate morning\nThe cry of a loon on a lake in the night\nthe dreams that are born in the dawn's early light The laughter that sings in the heart of a child\nThe freedom that flies at the call of the wild\